# 02 — DistilBERT Baseline Models

Train the proposal's real-data baselines: **B1** (real only), **B2** (random
oversampling), and **B3** (real plus EDA). Run notebook 01 first. This notebook uses
only the selected training subsets and shared validation; it never loads official test examples.

Apple silicon uses **MPS** automatically when available; Colab uses **CUDA**.
The default runs **B2 for the imbalanced setting and seeds 42, 43, 44**.
Balanced low-resource B2 is identical to B1 and reuses its recorded result without
another training job. `RUN_ALL_BASELINES = True` covers 18 logical conditions with
15 distinct training jobs. Scores here are **validation scores**, used for checkpoint selection.

Shared settings are loaded from `configs/distilbert_baseline.json`. Keep
`RUN_VALIDATION_CHECK = False` after the declared six-trial search is complete.


In [ ]:
import importlib
import importlib.metadata as metadata
import subprocess
import sys

required_packages = {
    "transformers": "5.19.0", "datasets": "5.1.0", "huggingface_hub": "2.2.0",
    "scikit-learn": "1.8.0", "nltk": "3.9.2",
}
install = []
loaded_to_update = []
for package, version in required_packages.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install.append(f"{package}=={version}")
        module = "sklearn" if package == "scikit-learn" else package
        if module in sys.modules:
            loaded_to_update.append(module)
try:
    metadata.version("torch")
except metadata.PackageNotFoundError:
    install.append("torch==2.14.1")

if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *install])
    importlib.invalidate_caches()
    if loaded_to_update:
        raise RuntimeError("Dependencies changed. Restart the kernel, then run all cells.")
print("Baseline dependencies are ready.")

## 1. Experiment settings and paths

Keep frozen model, optimizer, and training settings fixed across conditions. B2
preserves 1,315 distinct real imbalanced examples and appends 225 replacement draws:
1,540 training draws, or 20 per class. Its classifier uses ordinary cross-entropy.
B3 targets one EDA example per real example, with any shortfall recorded. Larger
training sets increase optimizer steps per epoch; steps and time are saved.

The validation-only check selected a shared learning rate using seed 42 in both
settings, with a 30-epoch ceiling and patience 5. Seeds 43/44 use the frozen decision
without further tuning. No training setting is changed for B2.

The default runs the three imbalanced B2 conditions. To select one condition, set
`RUN_B2_REPEATS = False`, then change `SELECTED_BASELINE`, `SELECTED_SETTING`, and
`SELECTED_SEED`. Only one of the batch-run flags may be enabled. Use a new
`EXPERIMENT_NAME` when changing hyperparameters. Matching completed runs are reused;
interrupted runs restart from the original seed and pretrained weights.


In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import time
from collections import Counter
from pathlib import Path

PROJECT_ROOT_OVERRIDE = None
EXPERIMENT_NAME = "distilbert_validated"
RUN_ALL_BASELINES = False
RUN_B1_REPEATS = False
RUN_B2_REPEATS = True
RUN_VALIDATION_CHECK = False
USE_FROZEN_CONFIG = True
TUNING_LEARNING_RATES = (2e-5, 5e-5, 1e-4)
TUNING_MAX_EPOCHS = 30
TUNING_PATIENCE = 5
TUNING_SEED = 42
SELECTED_BASELINE = "B2"
SELECTED_SETTING = "imbalanced"
SELECTED_SEED = 42
DEVICE_PREFERENCE = "auto"

MODEL_ID = "distilbert/distilbert-base-uncased"
MODEL_REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"
TRAINING_CONFIG = {
    "lr": 1e-4, "weight_decay": 0.01, "max_epochs": 30, "patience": 5,
    "batch_size": 16, "eval_batch_size": 32, "max_length": 128,
    "warmup_ratio": 0.1, "max_grad_norm": 1.0,
}
EDA_CONFIG = {
    "augmentations_per_real": 1, "attempts_per_example": 16,
    "alpha": 0.1, "deletion_probability": 0.1,
}

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/CS682_Neural_Project")
else:
    PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                         if (path / "notebooks").is_dir() and (path / "README.md").is_file()), None)
    if PROJECT_ROOT is None:
        raise RuntimeError("Set PROJECT_ROOT_OVERRIDE to the project directory.")

DATA_ROOT = PROJECT_ROOT / "data" / "reproducible"
RUN_ROOT = PROJECT_ROOT / "results" / "baselines" / EXPERIMENT_NAME
TUNING_ROOT = PROJECT_ROOT / "results" / "baselines" / "validation_check"
FROZEN_CONFIG_PATH = PROJECT_ROOT / "configs" / "distilbert_baseline.json"
MODEL_CACHE = DATA_ROOT / "_cache" / "hub"
NLTK_CACHE = DATA_ROOT / "_cache" / "nltk_data"
os.environ["HF_HOME"] = str(DATA_ROOT / "_cache")
os.environ["HF_HUB_CACHE"] = str(MODEL_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(DATA_ROOT / "_cache" / "datasets")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
print("Data:", DATA_ROOT)
print("Results:", RUN_ROOT)

## 2. Device and prepared-data checks

All backends use float32, the same pretrained checkpoint, and the same seed policy.
MPS may use a CPU fallback for unsupported operations. Runtime metadata records the
backend and versions; accelerator timings are synchronized elapsed intervals, not
hardware utilization measurements. Seeds do not guarantee bitwise equality across
MPS, CUDA, and CPU.

Prepared datasets are verified against notebook 01's content hashes before training.
The 770-example validation set is additional labeled supervision, kept separate
from the real training counts reported for each setting.

In [ ]:
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from datasets import load_from_disk
from transformers import AutoTokenizer


def require(condition, message):
    if not condition:
        raise ValueError(message)


def dataset_digest(data):
    digest = hashlib.sha256()
    digest.update(json.dumps(data.features.to_dict(), sort_keys=True).encode("utf-8"))
    for row in data:
        digest.update(json.dumps(row, sort_keys=True, ensure_ascii=False).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def rows_digest(rows):
    digest = hashlib.sha256()
    for row in rows:
        digest.update(json.dumps(dict(row), sort_keys=True, ensure_ascii=False).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


available_devices = {
    "cuda": torch.cuda.is_available(),
    "mps": torch.backends.mps.is_available(),
    "cpu": True,
}
if DEVICE_PREFERENCE == "auto":
    DEVICE = torch.device(next(name for name in ("cuda", "mps", "cpu") if available_devices[name]))
else:
    require(DEVICE_PREFERENCE in available_devices and available_devices[DEVICE_PREFERENCE],
            f"Requested device is unavailable: {DEVICE_PREFERENCE}")
    DEVICE = torch.device(DEVICE_PREFERENCE)
torch.set_num_threads(min(4, os.cpu_count() or 1))
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

manifest_path = DATA_ROOT / "manifests" / "experiment_manifest.json"
require(manifest_path.is_file(), "Run 01_reproducible_data_preparation.ipynb first.")
DATA_MANIFEST = json.loads(manifest_path.read_text())
CLASS_NAMES = DATA_MANIFEST["config"]["class_names"]
RARE_CLASS_IDS = DATA_MANIFEST["config"]["rare_class_ids"]
NUM_LABELS = len(CLASS_NAMES)
TRAINING_SEEDS = DATA_MANIFEST["config"]["training_seeds"]
require(NUM_LABELS == 77 and len(RARE_CLASS_IDS) == 15, "Unexpected label metadata.")
VALIDATION_DATA = load_from_disk(str(DATA_ROOT / "processed" / "validation_data"))
VALIDATION_DIGEST = dataset_digest(VALIDATION_DATA)
require(VALIDATION_DIGEST == DATA_MANIFEST["datasets"]["processed/validation_data"]["sha256"],
        "Validation data differs from notebook 01's manifest.")
require(Counter(VALIDATION_DATA["label"]) == Counter({label: 10 for label in range(NUM_LABELS)}),
        "Validation must contain 10 examples per class.")


def load_real_subset(setting, seed):
    require(setting in {"low_resource", "imbalanced"} and seed in TRAINING_SEEDS, "Unknown setting or seed.")
    name = f"processed/{setting}/seed_{seed}"
    data = load_from_disk(str(DATA_ROOT / name))
    digest = dataset_digest(data)
    require(digest == DATA_MANIFEST["datasets"][name]["sha256"], f"Training data differs from its manifest: {name}")
    require(not set(data["source_id"]) & set(VALIDATION_DATA["source_id"]), "Training/validation source overlap.")
    require(data.features["label"].names == CLASS_NAMES, "Training label mapping differs.")
    return data, digest


runtime_info = {
    "python": platform.python_version(), "platform": platform.platform(),
    "device": str(DEVICE), "precision": "float32",
    "mps_fallback": os.environ["PYTORCH_ENABLE_MPS_FALLBACK"],
    "packages": {name: metadata.version(name) for name in
                 ("torch", "transformers", "datasets", "huggingface_hub", "numpy", "pandas", "scikit-learn", "nltk")},
}
if DEVICE.type == "cuda":
    runtime_info["accelerator"] = torch.cuda.get_device_name(DEVICE)
elif DEVICE.type == "mps":
    runtime_info["accelerator"] = "Apple Metal Performance Shaders"
else:
    runtime_info["accelerator"] = platform.processor() or "CPU"

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, cache_dir=str(MODEL_CACHE), token=False)
print("Training device:", DEVICE, "| PyTorch:", torch.__version__, "| precision: float32")
display(pd.DataFrame([{"setting": setting, "seed": seed,
                       "rows": DATA_MANIFEST["datasets"][f"processed/{setting}/seed_{seed}"]["rows"]}
                      for setting in ("low_resource", "imbalanced") for seed in TRAINING_SEEDS]))

## 3. EDA for B3

EDA uses synonym replacement, random insertion, random swap, and random deletion,
following the [original EDA implementation](https://github.com/jasonwei20/eda_nlp).
This implementation protects numeric tokens and negation and retains punctuation.
Those safeguards are an adaptation; token edits can still change an intent.

Candidates are deduplicated against real training text and accepted EDA text only.
Validation and test text do not influence augmentation. A bounded retry budget avoids
forcing duplicates when a query has few usable edits. Requested and accepted counts
are saved per class. WordNet downloads only when B3 is selected.

In [ ]:
import random
import re
from collections import Counter
from functools import lru_cache


EDA_STOP_WORDS = frozenset("""i me my myself we our ours ourselves you your yours yourself yourselves he him his
himself she her hers herself it its itself they them their theirs themselves what
which who whom this that these those am is are was were be been being have has had
having do does did doing a an the and but if or because as until while of at by for
with about against between into through during before after above below to from up
down in out on off over under again further then once here there when where why how
all any both each few more most other some such no nor not only own same so than too
very can will just should now
""".split())
EDA_NEGATION = frozenset({"no", "not", "never", "nor", "neither", "cannot"})
EDA_METHODS = ("synonym_replacement", "random_insertion", "random_swap", "random_deletion")


def eda_tokens(text):
    return re.findall(r"\w+(?:['’]\w+)*(?:[.,:/-]\d+)*|[^\w\s]", text, flags=re.UNICODE)


def eda_normalize(text):
    return " ".join(token.casefold() for token in eda_tokens(text))


def eda_protected(token):
    word = token.casefold().replace("’", "'")
    return word in EDA_NEGATION or word.endswith("n't") or any(char.isdigit() for char in token)


def eda_join(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)
    text = re.sub(r"([$£€])\s+(?=\d)", r"\1", text)
    return text.strip()


@lru_cache(maxsize=10000)
def eda_synonyms(word):
    from nltk.corpus import wordnet

    choices = set()
    for synset in wordnet.synsets(word):
        for lemma in synset.lemmas():
            candidate = " ".join(lemma.name().replace("_", " ").replace("-", " ").casefold().split())
            if candidate != word.casefold() and candidate and all(part.isalpha() for part in candidate.split()):
                choices.add(candidate)
    # WordNet and set traversal order must not decide the augmentation seed.
    return tuple(sorted(choices))


def eda_variant(text, method, rng, alpha=0.1, deletion_probability=0.1):
    if alpha == 0 and method in EDA_METHODS[:3]:
        return text
    tokens = eda_tokens(text)
    eligible = [index for index, token in enumerate(tokens) if token.isalpha() and not eda_protected(token)]
    synonym_words = sorted({tokens[index].casefold() for index in eligible
                            if tokens[index].casefold() not in EDA_STOP_WORDS})
    n_edits = max(1, int(alpha * len([token for token in tokens if any(char.isalnum() for char in token)])))

    if method == "synonym_replacement":
        rng.shuffle(synonym_words)
        replaced = 0
        for word in synonym_words:
            choices = eda_synonyms(word)
            if choices:
                replacement = rng.choice(choices).split()
                tokens = [part for token in tokens for part in
                          (replacement if token.casefold() == word else [token])]
                replaced += 1
                if replaced == n_edits:
                    break
    elif method == "random_insertion":
        candidates = [(word, eda_synonyms(word)) for word in synonym_words]
        candidates = [(word, choices) for word, choices in candidates if choices]
        for _ in range(n_edits):
            if not candidates:
                break
            _, choices = rng.choice(candidates)
            position = rng.randrange(len(tokens) + 1)
            tokens[position:position] = rng.choice(choices).split()
    elif method == "random_swap":
        if len(eligible) >= 2:
            for _ in range(n_edits):
                first, second = rng.sample(eligible, 2)
                tokens[first], tokens[second] = tokens[second], tokens[first]
    elif method == "random_deletion":
        original = tokens
        tokens = [token for token in tokens if not token.isalpha() or eda_protected(token)
                  or rng.random() >= deletion_probability]
        if not any(any(char.isalnum() for char in token) for token in tokens):
            tokens = original
    else:
        raise ValueError(f"Unknown EDA method: {method}")
    return eda_join(tokens)


def build_eda_rows(real_data, seed, augmentations_per_real=1, attempts_per_example=16,
                   alpha=0.1, deletion_probability=0.1):
    if not isinstance(augmentations_per_real, int) or augmentations_per_real < 0:
        raise ValueError("augmentations_per_real must be a nonnegative integer")
    if not isinstance(attempts_per_example, int) or attempts_per_example < 1:
        raise ValueError("attempts_per_example must be a positive integer")
    if not 0 <= alpha <= 1 or not 0 <= deletion_probability <= 1:
        raise ValueError("EDA edit and deletion rates must lie in [0, 1]")

    real_rows = [dict(row) for row in real_data]
    rng = random.Random(seed)
    seen = {eda_normalize(row["text"]) for row in real_rows}
    records = [{**row, "is_augmented": False, "augmentation_id": f"real:{row['source_id']}",
                "augmentation_method": "original"} for row in real_rows]
    accepted_by_class = Counter()
    attempts_by_method = Counter({method: 0 for method in EDA_METHODS})
    accepted_by_method = Counter({method: 0 for method in EDA_METHODS})
    rejected = Counter()

    for row in real_rows:
        methods = list(EDA_METHODS)
        rng.shuffle(methods)
        accepted = 0
        attempt = 0
        while accepted < augmentations_per_real and attempt < attempts_per_example * augmentations_per_real:
            method = methods[attempt % len(methods)]
            candidate = eda_variant(row["text"], method, rng, alpha, deletion_probability)
            attempt += 1
            attempts_by_method[method] += 1
            normalized = eda_normalize(candidate)
            if not normalized or not any(char.isalnum() for char in candidate):
                rejected["empty"] += 1
                continue
            if normalized == eda_normalize(row["text"]):
                rejected["unchanged"] += 1
                continue
            if normalized in seen:
                rejected["duplicate"] += 1
                continue
            seen.add(normalized)
            records.append({**row, "text": candidate, "is_augmented": True,
                            "augmentation_id": f"eda:{seed}:{row['source_id']}:{accepted}",
                            "augmentation_method": method})
            accepted += 1
            accepted_by_class[int(row["label"])] += 1
            accepted_by_method[method] += 1

    original_counts = Counter(int(row["label"]) for row in real_rows)
    requested = len(real_rows) * augmentations_per_real
    accepted = sum(accepted_by_class.values())
    report = {
        "seed": seed,
        "alpha": alpha,
        "deletion_probability": deletion_probability,
        "augmentations_per_real": augmentations_per_real,
        "attempts_per_example": attempts_per_example,
        "real_rows": len(real_rows),
        "requested_augmented_rows": requested,
        "accepted_augmented_rows": accepted,
        "shortfall": requested - accepted,
        "training_rows": len(records),
        "attempts_by_method": dict(attempts_by_method),
        "accepted_by_method": dict(accepted_by_method),
        "rejections": dict(rejected),
        "per_class": {str(label): {"requested": count * augmentations_per_real,
                                   "accepted": accepted_by_class[label],
                                   "shortfall": count * augmentations_per_real - accepted_by_class[label]}
                      for label, count in sorted(original_counts.items())},
        "policy": "SR/RI/RS/RD; preserve punctuation, negation and numeric tokens; deduplicate against real training and accepted EDA only",
    }
    return records, report

## 4. Training and validation helpers

B2 uses seeded random oversampling to the largest real class count. It preserves
every original row and duplicates only selected training examples; text and labels
are unchanged. Repeated source IDs are expected draws, not new unique examples.
Oversampling and class-weighted loss are not combined. In balanced low-resource
data, no draw is added, so B2 reuses B1 as an identity condition.

Training and validation loss use ordinary cross-entropy. Macro-F1 averages all 77
classes; rare-class metrics average the fixed 15 rare labels while accounting for
predictions on all validation rows. Each condition starts from the same model and
seed, uses the frozen hyperparameters, and selects its checkpoint by validation
macro-F1. Best weights, tokenizer, history, class scores, probabilities, provenance,
and timing logs are saved. B1 remains the real-data classifier for F2 later.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, recall_score
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup


def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


def release_device_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()


def seed_training(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if DEVICE.type == "mps":
        torch.mps.manual_seed(seed)


def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def write_csv_atomic(path, rows):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    pd.DataFrame(rows).to_csv(temporary, index=False)
    os.replace(temporary, path)


def validation_metrics(labels, predictions):
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, labels=list(range(NUM_LABELS)), average="macro", zero_division=0)),
        "rare_macro_f1": float(f1_score(labels, predictions, labels=RARE_CLASS_IDS, average="macro", zero_division=0)),
        "rare_macro_recall": float(recall_score(labels, predictions, labels=RARE_CLASS_IDS, average="macro", zero_division=0)),
    }


def tokenize_rows(rows):
    encoded = TOKENIZER([row["text"] for row in rows], truncation=True, max_length=TRAINING_CONFIG["max_length"])
    features = [{**{key: values[index] for key, values in encoded.items()}, "labels": int(row["label"])}
                for index, row in enumerate(rows)]
    return features, sum(len(ids) for ids in encoded["input_ids"])


def evaluate_classifier(model, loader):
    model.eval()
    labels, probabilities, loss_sum, count = [], [], 0.0, 0
    synchronize_device()
    started = time.perf_counter()
    with torch.inference_mode():
        for batch in loader:
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            gold = batch.pop("labels")
            logits = model(**batch).logits
            loss_sum += float(torch.nn.functional.cross_entropy(logits, gold, reduction="sum").cpu())
            count += len(gold)
            labels.extend(gold.cpu().tolist())
            probabilities.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    synchronize_device()
    elapsed = time.perf_counter() - started
    probabilities = np.concatenate(probabilities)
    labels = np.asarray(labels, dtype=np.int64)
    predictions = probabilities.argmax(axis=1)
    metrics = {"validation_loss": loss_sum / count, **validation_metrics(labels, predictions)}
    return metrics, labels, predictions, probabilities, elapsed


def run_baseline(setting, seed, condition, real_data, train_rows, eda_report, data_signature, balancing_report=None):
    if condition not in {"B1", "B2", "B3"}:
        raise ValueError(f"Unknown baseline: {condition}")
    real_rows, train_rows, validation_rows = list(real_data), list(train_rows), list(VALIDATION_DATA)
    if not real_rows or not train_rows or not validation_rows:
        raise ValueError("Training and validation data must be nonempty.")
    if set(row["label"] for row in real_rows) != set(range(NUM_LABELS)):
        raise ValueError("The real training subset must contain every intent class.")
    if any(not 0 <= int(row["label"]) < NUM_LABELS for row in train_rows + validation_rows):
        raise ValueError("A row contains an invalid intent label.")
    if any(TRAINING_CONFIG[key] < 1 for key in ("max_epochs", "patience", "batch_size", "eval_batch_size")):
        raise ValueError("Epochs, patience, and batch sizes must be positive.")
    run_dir = RUN_ROOT / setting / condition / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    best_path, config_path, result_path = run_dir / "best_model", run_dir / "run_config.json", run_dir / "results.json"
    specification = {
        "schema_version": 1, "training_code_version": 2, "setting": setting, "seed": int(seed), "condition": condition,
        "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "training": dict(TRAINING_CONFIG),
        "data_signature": data_signature, "class_names": list(CLASS_NAMES), "rare_class_ids": list(RARE_CLASS_IDS),
        "device": str(DEVICE), "precision": "float32", "runtime": runtime_info,
    }
    if condition == "B2":
        require(balancing_report is not None, "B2 requires a balancing report.")
        require(balancing_report["method"] == "random_oversampling", "B2 method must be random oversampling.")
        require(balancing_report["real_unique_rows"] == len(real_rows)
                and balancing_report["training_draws"] == len(train_rows), "B2 row counts differ from their report.")
        require(balancing_report["seed"] == seed, "B2 sampling seed differs.")
        expected_balancing = {
            key: balancing_report[key] for key in
            ("method", "implementation_version", "seed", "target_per_class", "real_unique_rows", "training_draws", "additional_draws")
        }
        require(data_signature.get("balancing") == expected_balancing, "B2 signature differs from its balancing report.")
        specification["b2_variant"] = {
            "method": "random_oversampling", "implementation_version": 1,
            "loss": "ordinary_cross_entropy",
        }
    if config_path.exists():
        previous_specification = json.loads(config_path.read_text())
        execution_fields = {"device", "runtime"}
        previous_core = {key: value for key, value in previous_specification.items() if key not in execution_fields}
        current_core = {key: value for key, value in specification.items() if key not in execution_fields}
        if previous_core != current_core:
            raise ValueError(f"Run configuration differs at {run_dir}; choose another RUN_ROOT.")
        if previous_specification != specification:
            if not result_path.exists():
                raise ValueError(f"Interrupted run has a different runtime at {run_dir}; choose another RUN_ROOT.")
            print("Using the completed result's recorded hardware and package versions; the current runtime differs.", flush=True)
    elif any(run_dir.iterdir()):
        raise ValueError(f"Existing run has no configuration: {run_dir}")
    else:
        write_json_atomic(config_path, specification)
    if condition == "B2":
        balancing_path = run_dir / "balancing_report.json"
        if balancing_path.exists():
            require(json.loads(balancing_path.read_text()) == balancing_report, "Saved B2 balancing report differs.")
        else:
            require(not result_path.exists(), "Completed B2 run is missing its balancing report.")
            write_json_atomic(balancing_path, balancing_report)
    if result_path.exists():
        completed = json.loads(result_path.read_text())
        required = [best_path / "config.json", best_path / "tokenizer_config.json",
                    run_dir / "validation_predictions.csv", run_dir / "validation_probabilities.npz",
                    run_dir / "per_class_metrics.csv", run_dir / "history.csv", run_dir / "stage_log.csv"]
        if completed.get("status") != "completed" or not all(path.is_file() for path in required) or not list(best_path.glob("*.safetensors")):
            raise ValueError(f"Completed run has missing artifacts: {run_dir}")
        print(f"Reusing completed {setting}/{condition}/seed_{seed}: macro-F1={completed['macro_f1']:.4f}", flush=True)
        return completed
    if (run_dir / "history.csv").exists() or best_path.exists():
        print(f"Restarting interrupted {setting}/{condition}/seed_{seed} from its original seed and pretrained weights.", flush=True)

    started = time.perf_counter()
    seed_training(seed)
    train_features, train_tokens = tokenize_rows(train_rows)
    validation_features, validation_tokens = tokenize_rows(validation_rows)
    collator = DataCollatorWithPadding(TOKENIZER, return_tensors="pt")
    train_loader = DataLoader(train_features, batch_size=TRAINING_CONFIG["batch_size"], shuffle=True,
                              generator=torch.Generator().manual_seed(seed), collate_fn=collator, num_workers=0)
    validation_loader = DataLoader(validation_features, batch_size=TRAINING_CONFIG["eval_batch_size"],
                                   shuffle=False, collate_fn=collator, num_workers=0)
    model = optimizer = scheduler = None
    history, stages, best_score, best_epoch, stale_epochs, steps = [], [], -1.0, 0, 0, 0
    training_seconds = validation_seconds = 0.0
    augmentation_seconds = float(eda_report.get("augmentation_seconds", 0.0)) if eda_report else 0.0
    if eda_report:
        stages.append({"stage": "eda", "epoch": 0, "setting": setting, "condition": condition, "seed": seed,
                       "device": "cpu", "wall_seconds": augmentation_seconds, "accelerator_synchronized_wall_seconds": 0.0,
                       "nonpadding_tokens": 0, "optimizer_steps": 0})
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION, cache_dir=str(MODEL_CACHE), token=False,
            num_labels=NUM_LABELS, id2label={index: name for index, name in enumerate(CLASS_NAMES)},
            label2id={name: index for index, name in enumerate(CLASS_NAMES)}, attn_implementation="eager",
        ).float().to(DEVICE)
        class_counts = np.bincount([row["label"] for row in real_rows], minlength=NUM_LABELS)
        weights = None
        criterion = torch.nn.CrossEntropyLoss(weight=weights)
        groups = [{"params": [parameter for parameter in model.parameters() if parameter.requires_grad and parameter.ndim >= 2],
                   "weight_decay": TRAINING_CONFIG["weight_decay"]},
                  {"params": [parameter for parameter in model.parameters() if parameter.requires_grad and parameter.ndim < 2], "weight_decay": 0.0}]
        optimizer = torch.optim.AdamW(groups, lr=TRAINING_CONFIG["lr"], foreach=False)
        total_steps = TRAINING_CONFIG["max_epochs"] * len(train_loader)
        scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * TRAINING_CONFIG["warmup_ratio"]), total_steps)
        for epoch in range(1, TRAINING_CONFIG["max_epochs"] + 1):
            model.train()
            loss_sum, loss_denominator = 0.0, 0.0
            synchronize_device()
            epoch_started = time.perf_counter()
            for batch in train_loader:
                batch = {key: value.to(DEVICE) for key, value in batch.items()}
                labels = batch.pop("labels")
                optimizer.zero_grad(set_to_none=True)
                logits = model(**batch).logits
                loss = criterion(logits, labels)
                if not bool(torch.isfinite(loss).cpu()):
                    raise RuntimeError("Training loss is non-finite; no completion result was saved.")
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), TRAINING_CONFIG["max_grad_norm"])
                optimizer.step()
                scheduler.step()
                denominator = float(weights[labels].sum().cpu()) if weights is not None else len(labels)
                loss_sum += float(loss.detach().cpu()) * denominator
                loss_denominator += denominator
                steps += 1
            synchronize_device()
            train_elapsed = time.perf_counter() - epoch_started
            metrics, _, _, _, validation_elapsed = evaluate_classifier(model, validation_loader)
            training_seconds += train_elapsed
            validation_seconds += validation_elapsed
            history.append({"epoch": epoch, "train_loss": loss_sum / loss_denominator, **metrics,
                            "training_seconds": train_elapsed, "validation_seconds": validation_elapsed,
                            "optimizer_steps": steps, "learning_rate": scheduler.get_last_lr()[0]})
            for stage, elapsed, tokens in (("training", train_elapsed, train_tokens), ("validation", validation_elapsed, validation_tokens)):
                stages.append({"stage": stage, "epoch": epoch, "setting": setting, "condition": condition, "seed": seed,
                               "device": str(DEVICE), "wall_seconds": elapsed,
                               "accelerator_synchronized_wall_seconds": elapsed if DEVICE.type != "cpu" else 0.0,
                               "nonpadding_tokens": tokens, "optimizer_steps": len(train_loader) if stage == "training" else 0})
            write_csv_atomic(run_dir / "history.csv", history)
            write_csv_atomic(run_dir / "stage_log.csv", stages)
            print(f"{setting}/{condition}/seed_{seed} epoch {epoch}: val macro-F1={metrics['macro_f1']:.4f}, train={train_elapsed:.1f}s", flush=True)
            if metrics["macro_f1"] > best_score:
                best_score, best_epoch, stale_epochs = metrics["macro_f1"], epoch, 0
                model.save_pretrained(str(best_path), safe_serialization=True)
                TOKENIZER.save_pretrained(str(best_path))
            else:
                stale_epochs += 1
                if stale_epochs >= TRAINING_CONFIG["patience"]:
                    break
        del batch, labels, logits, loss, criterion, weights, groups
        model = optimizer = scheduler = None
        release_device_memory()
        model = AutoModelForSequenceClassification.from_pretrained(str(best_path), local_files_only=True, attn_implementation="eager").float().to(DEVICE)
        metrics, labels, predictions, probabilities, elapsed = evaluate_classifier(model, validation_loader)
        validation_seconds += elapsed
        stages.append({"stage": "best_checkpoint_validation", "epoch": best_epoch, "setting": setting,
                       "condition": condition, "seed": seed, "device": str(DEVICE), "wall_seconds": elapsed,
                       "accelerator_synchronized_wall_seconds": elapsed if DEVICE.type != "cpu" else 0.0,
                       "nonpadding_tokens": validation_tokens, "optimizer_steps": 0})
        if not np.isclose(metrics["macro_f1"], best_score):
            raise RuntimeError("Reloaded checkpoint metrics differ from its recorded best epoch.")
        write_csv_atomic(run_dir / "validation_predictions.csv", [
            {"source_id": row["source_id"], "text": row["text"], "true_label": int(label), "predicted_label": int(prediction),
             "true_intent": CLASS_NAMES[int(label)], "predicted_intent": CLASS_NAMES[int(prediction)], "confidence": float(probabilities[index].max())}
            for index, (row, label, prediction) in enumerate(zip(validation_rows, labels, predictions))])
        probability_path = run_dir / "validation_probabilities.npz"
        with probability_path.with_name(probability_path.name + ".tmp").open("wb") as stream:
            np.savez_compressed(stream, probabilities=probabilities, labels=labels,
                                source_ids=np.asarray([row["source_id"] for row in validation_rows]))
        os.replace(probability_path.with_name(probability_path.name + ".tmp"), probability_path)
        precision, recall, f1, support = precision_recall_fscore_support(labels, predictions, labels=list(range(NUM_LABELS)), zero_division=0)
        write_csv_atomic(run_dir / "per_class_metrics.csv", [
            {"label": label, "intent": CLASS_NAMES[label], "rare_class": label in RARE_CLASS_IDS,
             "precision": float(precision[label]), "recall": float(recall[label]), "f1": float(f1[label]), "support": int(support[label])}
            for label in range(NUM_LABELS)])
        write_csv_atomic(run_dir / "stage_log.csv", stages)
        classifier_seconds = time.perf_counter() - started
        result = {"status": "completed", "setting": setting, "seed": int(seed), "condition": condition,
                  **metrics, "best_epoch": best_epoch, "epochs_run": len(history), "real_rows": len(real_rows), "train_rows": len(train_rows),
                  "optimizer_steps": steps, "training_tokens": train_tokens * len(history),
                  "validation_tokens": validation_tokens * (len(history) + 1), "training_seconds": training_seconds,
                  "validation_seconds": validation_seconds, "classifier_seconds": classifier_seconds,
                  "augmentation_seconds": augmentation_seconds, "total_seconds": classifier_seconds + augmentation_seconds,
                  "timing_note": "Accelerator timings are synchronized wall intervals, not measured utilization. Total includes original EDA generation time, including when cached EDA is reused.",
                  "device": str(DEVICE), "precision": "float32", "model_path": str(best_path), "run_path": str(run_dir), "eda": eda_report}
        if condition == "B2":
            result.update({
                "balancing_method": "random_oversampling",
                "real_unique_rows": balancing_report["real_unique_rows"],
                "training_draws": balancing_report["training_draws"],
                "additional_draws": balancing_report["additional_draws"],
                "balancing_report_path": str(balancing_path),
            })
        write_json_atomic(result_path, result)
        return result
    finally:
        model = optimizer = scheduler = None
        release_device_memory()

## 5. Validate and freeze shared settings

`RUN_VALIDATION_CHECK` controls a fixed search: learning rates 2e-5, 5e-5, and 1e-4;
seed 42; both low-resource and imbalanced training; at most 30 epochs; early stopping
after five epochs without improved validation macro-F1. The lower rate wins when mean
scores differ by at most 0.001. Model architecture, data, batch size, and other settings
stay fixed. The official test data is never loaded.

Each best checkpoint is also evaluated on its own training subset to check whether it
fits the available examples. The duration diagnostic compares the first 10 epochs with
the full curve **under the same 30-epoch schedule**; it is not a reproduction of the old
10-epoch learning-rate decay. These selected validation scores are development results.

The decision is saved to `configs/distilbert_baseline.json`; small CSV reports go to
`results/metrics/`. Tuning compute is recorded separately. Selected seed-42 B1 weights
are copied into the baseline experiment and reused, so they are trained once. Keep
`RUN_VALIDATION_CHECK=False` for the remaining baseline and synthetic experiments.


In [ ]:
import shutil


def choose_shared_learning_rate(records, learning_rates, settings, tolerance=0.001):
    expected = {(float(rate), setting) for rate in learning_rates for setting in settings}
    observed = [(float(row["learning_rate"]), row["setting"]) for row in records]
    require(len(observed) == len(expected) and set(observed) == expected,
            "Every learning rate must have exactly one completed trial in each setting.")
    table = pd.DataFrame(records).groupby("learning_rate", as_index=False).agg(
        mean_macro_f1=("macro_f1", "mean"), minimum_macro_f1=("macro_f1", "min"),
        mean_accuracy=("accuracy", "mean"), mean_best_epoch=("best_epoch", "mean"),
    ).sort_values("learning_rate")
    maximum = float(table["mean_macro_f1"].max())
    selected = float(table.loc[table["mean_macro_f1"] >= maximum - tolerance, "learning_rate"].min())
    return selected, table


def check_training_fit(real_data, folder):
    diagnostic_started = time.perf_counter()
    diagnostic_path = folder / "training_fit.json"
    specification = {
        "training_data_sha256": dataset_digest(real_data),
        "run_config_sha256": hashlib.sha256((folder / "run_config.json").read_bytes()).hexdigest(),
    }
    if diagnostic_path.exists():
        saved = json.loads(diagnostic_path.read_text())
        require(saved["source"] == specification, "Training-fit diagnostic source changed.")
        return saved
    model = None
    try:
        features, _ = tokenize_rows(list(real_data))
        loader = DataLoader(features, batch_size=TRAINING_CONFIG["eval_batch_size"], shuffle=False,
                            collate_fn=DataCollatorWithPadding(TOKENIZER, return_tensors="pt"), num_workers=0)
        model = AutoModelForSequenceClassification.from_pretrained(
            str(folder / "best_model"), local_files_only=True, attn_implementation="eager",
        ).float().to(DEVICE)
        metrics, _, _, _, elapsed = evaluate_classifier(model, loader)
        diagnostic = {
            "source": specification, "training_loss": metrics["validation_loss"],
            "training_accuracy": metrics["accuracy"], "training_macro_f1": metrics["macro_f1"],
            "inference_seconds": elapsed, "diagnostic_seconds": time.perf_counter() - diagnostic_started,
        }
        write_json_atomic(diagnostic_path, diagnostic)
        return diagnostic
    finally:
        model = None
        release_device_memory()


def reuse_selected_b1(selection):
    for setting, relative_source in selection["selected_trial_directories"].items():
        source = TUNING_ROOT / relative_source
        destination = RUN_ROOT / setting / "B1" / f"seed_{TUNING_SEED}"
        require(source.resolve() != destination.resolve(), "Tuning and final baseline directories must differ.")
        source_config = json.loads((source / "run_config.json").read_text())
        if destination.exists():
            require((destination / "results.json").is_file(), f"Incomplete baseline directory: {destination}")
            require(json.loads((destination / "run_config.json").read_text()) == source_config,
                    f"Existing baseline configuration differs: {destination}")
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, destination)
        result = json.loads((destination / "results.json").read_text())
        result.update({
            "model_path": str(destination / "best_model"), "run_path": str(destination),
            "training_reused_from_tuning": relative_source, "additional_training_seconds": 0.0,
            "accounting_note": "Selected tuning weights reused. Count their training compute once in project totals.",
        })
        write_json_atomic(destination / "results.json", result)


def run_validation_check():
    global TRAINING_CONFIG, RUN_ROOT
    settings = ("low_resource", "imbalanced")
    original_config, original_root = dict(TRAINING_CONFIG), RUN_ROOT
    fixed = {key: value for key, value in original_config.items() if key not in {"lr", "max_epochs", "patience"}}
    manifest_sha = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    plan = {
        "schema_version": 1, "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
        "seed": TUNING_SEED, "settings": list(settings), "learning_rates": list(TUNING_LEARNING_RATES),
        "max_epochs": TUNING_MAX_EPOCHS, "patience": TUNING_PATIENCE, "fixed_training_parameters": fixed,
        "data_manifest_sha256": manifest_sha, "validation_sha256": VALIDATION_DIGEST,
        "selection_metric": "equal-weight mean of the two settings' best validation macro-F1",
        "tie_policy": "choose the lower learning rate within 0.001 macro-F1 of the maximum mean",
        "duration_comparison": f"first 10 epochs versus full curve under the same {TUNING_MAX_EPOCHS}-epoch decay schedule",
        "official_test_used": False,
    }
    TUNING_ROOT.mkdir(parents=True, exist_ok=True)
    plan_path = TUNING_ROOT / "search_plan.json"
    if plan_path.exists():
        require(json.loads(plan_path.read_text()) == plan, "Search plan differs; choose a new TUNING_ROOT.")
    else:
        require(not any(TUNING_ROOT.iterdir()), "Tuning output directory has files but no search plan.")
        write_json_atomic(plan_path, plan)
    records = []
    try:
        for rate in TUNING_LEARNING_RATES:
            TRAINING_CONFIG = {**fixed, "lr": rate, "max_epochs": TUNING_MAX_EPOCHS, "patience": TUNING_PATIENCE}
            RUN_ROOT = TUNING_ROOT / f"lr_{rate:.0e}"
            for setting in settings:
                real_data, digest = load_real_subset(setting, TUNING_SEED)
                signature = {"real_data_sha256": digest, "validation_sha256": VALIDATION_DIGEST,
                             "actual_training_rows_sha256": rows_digest(list(real_data)), "eda_config": None}
                result = run_baseline(setting, TUNING_SEED, "B1", real_data, list(real_data), None, signature)
                folder = RUN_ROOT / setting / "B1" / f"seed_{TUNING_SEED}"
                fit = check_training_fit(real_data, folder)
                history = pd.read_csv(folder / "history.csv")
                prefix = history.loc[history["epoch"] <= 10]
                first_ten = float(prefix["macro_f1"].max())
                records.append({
                    "learning_rate": rate, "setting": setting, "seed": TUNING_SEED,
                    "macro_f1": result["macro_f1"], "accuracy": result["accuracy"],
                    "rare_macro_f1": result["rare_macro_f1"], "best_epoch": result["best_epoch"],
                    "epochs_run": result["epochs_run"], "first_10_epoch_best_macro_f1": first_ten,
                    "later_epoch_gain": result["macro_f1"] - first_ten,
                    "training_loss": fit["training_loss"], "training_accuracy": fit["training_accuracy"],
                    "training_macro_f1": fit["training_macro_f1"],
                    "tuning_seconds": result["total_seconds"] + fit["diagnostic_seconds"],
                    "device": result["device"], "trial_directory": str(folder.relative_to(TUNING_ROOT)),
                })
                write_csv_atomic(TUNING_ROOT / "trial_summary.csv", records)
                print(f"Trial complete: lr={rate:g}, {setting}; validation F1={result['macro_f1']:.4f}, training accuracy={fit['training_accuracy']:.4f}", flush=True)
        selected_rate, score_table = choose_shared_learning_rate(records, TUNING_LEARNING_RATES, settings)
        score_table.to_csv(TUNING_ROOT / "learning_rate_summary.csv", index=False)
        report_directory = PROJECT_ROOT / "results" / "metrics"
        report_directory.mkdir(parents=True, exist_ok=True)
        write_csv_atomic(report_directory / "baseline_validation_trials.csv", records)
        score_table.to_csv(report_directory / "baseline_learning_rates.csv", index=False)
        selection = {
            "schema_version": 1, "status": "frozen", "search_plan": plan,
            "search_plan_sha256": hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest(),
            "data_manifest_sha256": manifest_sha, "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
            "training_config": {**fixed, "lr": selected_rate, "max_epochs": TUNING_MAX_EPOCHS, "patience": TUNING_PATIENCE},
            "selected_learning_rate": selected_rate,
            "selected_mean_macro_f1": float(score_table.loc[score_table["learning_rate"] == selected_rate, "mean_macro_f1"].iloc[0]),
            "selected_trial_directories": {row["setting"]: row["trial_directory"] for row in records if row["learning_rate"] == selected_rate},
            "completed_trials": len(records), "total_tuning_seconds": sum(row["tuning_seconds"] for row in records),
            "development_scores_only": True, "official_test_used": False,
            "confirmation_policy": "Seeds 43 and 44 use these settings without further hyperparameter search.",
            "cost_policy": "Report search compute separately and count reused selected B1 training only once.",
        }
        if FROZEN_CONFIG_PATH.exists():
            previous = json.loads(FROZEN_CONFIG_PATH.read_text())
            require(previous["search_plan_sha256"] == selection["search_plan_sha256"]
                    and previous["training_config"] == selection["training_config"],
                    "Existing frozen decision differs; do not silently replace it.")
            selection = previous
        else:
            FROZEN_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
            write_json_atomic(FROZEN_CONFIG_PATH, selection)
        display(pd.DataFrame(records))
        display(score_table)
        print("Frozen shared training configuration:", selection["training_config"], flush=True)
        return selection
    finally:
        TRAINING_CONFIG, RUN_ROOT = original_config, original_root


if RUN_VALIDATION_CHECK:
    selection = run_validation_check()
    reuse_selected_b1(selection)

if USE_FROZEN_CONFIG and FROZEN_CONFIG_PATH.exists():
    selection = json.loads(FROZEN_CONFIG_PATH.read_text())
    require(selection["status"] == "frozen" and selection["official_test_used"] is False, "Invalid frozen decision.")
    require(selection["data_manifest_sha256"] == hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
            "Frozen settings were selected with different data.")
    require(selection["model_id"] == MODEL_ID and selection["model_revision"] == MODEL_REVISION,
            "Frozen settings use a different pretrained model.")
    TRAINING_CONFIG = dict(selection["training_config"])
    print("Using frozen shared settings:", TRAINING_CONFIG, flush=True)
    if not RUN_VALIDATION_CHECK:
        report_directory = PROJECT_ROOT / "results" / "metrics"
        for filename in ("baseline_validation_trials.csv", "baseline_learning_rates.csv"):
            report = report_directory / filename
            if report.exists():
                display(pd.read_csv(report))
elif USE_FROZEN_CONFIG:
    raise RuntimeError("Set RUN_VALIDATION_CHECK=True to create the frozen decision, or USE_FROZEN_CONFIG=False for a separate manual experiment.")


## 6. Prepare balancing or augmentation and run experiments

B2 keeps the original selected rows first, then appends seeded replacement draws
until each class reaches its largest original class count. Its report records
unique originals, training draws, per-class counts, and source multiplicities.
`balancing_report.json` is saved in each B2 run folder. No validation text is used.
Balanced low-resource B2 is recorded in `balanced_b2_identity.csv` as a B1 identity,
with zero additional training time and no extra trained model.

EDA outputs are cached with source hashes, settings, WordNet version, and output
hashes. Completed classifier runs require matching saved configurations and
artifacts. Changing B2's method changes only B2 metadata; the completed B1 cache
configuration remains compatible. All conditions keep the same frozen settings.


In [ ]:
from collections import Counter, defaultdict
import random


def build_random_oversampling(real_data, seed, num_labels=77):
    original = [dict(row) for row in real_data]
    if not original:
        raise ValueError("Cannot oversample an empty training subset.")
    source_rows = {row["source_id"]: row for row in original}
    if len(source_rows) != len(original):
        raise ValueError("The original training subset must have unique source IDs.")
    groups = defaultdict(list)
    for row in original:
        groups[int(row["label"])].append(row)
    if set(groups) != set(range(num_labels)):
        raise ValueError("Every intent class must be represented in the real subset.")
    target = max(len(rows) for rows in groups.values())
    rng = random.Random(seed)
    draws = [dict(row) for row in original]
    for label in range(num_labels):
        draws.extend(dict(rng.choice(groups[label])) for _ in range(target - len(groups[label])))

    counts_before = Counter(int(row["label"]) for row in original)
    counts_after = Counter(int(row["label"]) for row in draws)
    if counts_after != Counter({label: target for label in range(num_labels)}):
        raise ValueError("Oversampling did not produce the target class counts.")
    if draws[:len(original)] != original:
        raise ValueError("Oversampling altered the original training rows.")
    if set(row["source_id"] for row in draws) != set(source_rows):
        raise ValueError("Oversampling changed the set of unique real training rows.")
    if any(row != source_rows[row["source_id"]] for row in draws):
        raise ValueError("An oversampling draw differs from its original source row.")
    multiplicities = Counter(row["source_id"] for row in draws)
    report = {
        "method": "random_oversampling",
        "implementation_version": 1,
        "seed": int(seed),
        "target_per_class": target,
        "real_unique_rows": len(original),
        "training_draws": len(draws),
        "additional_draws": len(draws) - len(original),
        "no_op": len(draws) == len(original),
        "class_counts_before": {str(label): counts_before[label] for label in range(num_labels)},
        "class_counts_after": {str(label): counts_after[label] for label in range(num_labels)},
        "source_multiplicities": dict(sorted(multiplicities.items())),
        "policy": "Preserve every selected real row; append seeded replacement draws to the largest class count; use ordinary cross-entropy.",
    }
    return draws, report


def record_balanced_b2_identities():
    aliases = []
    for seed in TRAINING_SEEDS:
        folder = RUN_ROOT / "low_resource" / "B1" / f"seed_{seed}"
        if not (folder / "results.json").is_file():
            continue
        real_data, real_digest = load_real_subset("low_resource", seed)
        draws, report = build_random_oversampling(real_data, seed, NUM_LABELS)
        require(report["no_op"] and draws == list(real_data), "Balanced B2 must be identical to B1.")
        expected_signature = {
            "real_data_sha256": real_digest, "validation_sha256": VALIDATION_DIGEST,
            "actual_training_rows_sha256": rows_digest(list(real_data)), "eda_config": None,
        }
        recorded_config = json.loads((folder / "run_config.json").read_text())
        require(recorded_config["training"] == TRAINING_CONFIG
                and recorded_config["model_id"] == MODEL_ID
                and recorded_config["model_revision"] == MODEL_REVISION
                and recorded_config["data_signature"] == expected_signature,
                "Balanced B1 source differs from the current frozen experiment.")
        result = json.loads((folder / "results.json").read_text())
        require(result["status"] == "completed", "Balanced B1 source has not completed.")
        require((folder / "best_model" / "config.json").is_file()
                and bool(list((folder / "best_model").glob("*.safetensors"))), "Balanced B1 checkpoint is missing.")
        aliases.append({
            "setting": "low_resource", "condition": "B2", "seed": seed,
            "balancing_method": "random_oversampling", "status": "reused_B1_identity",
            "unique_training_job": False, "additional_training_seconds": 0.0,
            "real_unique_rows": len(real_data), "training_draws": len(draws),
            "macro_f1": result["macro_f1"], "accuracy": result["accuracy"],
            "rare_macro_f1": result["rare_macro_f1"], "rare_macro_recall": result["rare_macro_recall"],
            "source_result_path": str(folder / "results.json"), "model_path": str(folder / "best_model"),
        })
    if aliases:
        write_csv_atomic(RUN_ROOT / "balanced_b2_identity.csv", aliases)
    return aliases


def prepare_eda(real_data, setting, seed, source_digest):
    import nltk
    from nltk.corpus import wordnet

    NLTK_CACHE.mkdir(parents=True, exist_ok=True)
    if str(NLTK_CACHE) not in nltk.data.path:
        nltk.data.path.insert(0, str(NLTK_CACHE))
    try:
        wordnet.ensure_loaded()
    except LookupError:
        require(nltk.download("wordnet", download_dir=str(NLTK_CACHE), quiet=True), "WordNet download failed.")
        wordnet.ensure_loaded()
    specification = {
        "implementation_version": 1, "setting": setting, "seed": seed, "real_data_sha256": source_digest,
        "wordnet_version": wordnet.get_version(), "nltk_version": metadata.version("nltk"), **EDA_CONFIG,
    }
    folder = RUN_ROOT / "eda" / setting / f"seed_{seed}"
    config_path, rows_path, report_path = folder / "config.json", folder / "training_rows.json", folder / "report.json"
    folder.mkdir(parents=True, exist_ok=True)
    if config_path.exists():
        require(json.loads(config_path.read_text()) == specification, "EDA configuration differs; choose a new experiment name.")
    elif any(folder.iterdir()):
        raise ValueError(f"Existing EDA folder lacks configuration: {folder}")
    else:
        write_json_atomic(config_path, specification)

    if rows_path.exists() and report_path.exists():
        rows, report = json.loads(rows_path.read_text()), json.loads(report_path.read_text())
        require(rows_digest(rows) == report["training_rows_sha256"], "Cached EDA rows fail their content check.")
    else:
        started = time.perf_counter()
        rows, report = build_eda_rows(real_data, seed, **EDA_CONFIG)
        report.update({"augmentation_seconds": time.perf_counter() - started,
                       "training_rows_sha256": rows_digest(rows), "wordnet_version": wordnet.get_version()})
        write_json_atomic(rows_path, rows)
        write_json_atomic(report_path, report)

    original = list(real_data)
    require([{key: row[key] for key in original[0]} for row in rows[:len(original)]] == original,
            "EDA did not preserve the original real examples.")
    label_by_source = {row["source_id"]: row["label"] for row in original}
    require(all(row["source_id"] in label_by_source and row["label"] == label_by_source[row["source_id"]] for row in rows),
            "EDA provenance or intended labels differ from the selected training data.")
    require(len({row["augmentation_id"] for row in rows}) == len(rows), "Repeated augmentation IDs.")
    print(f"EDA {setting}/seed_{seed}: {report['accepted_augmented_rows']}/{report['requested_augmented_rows']} accepted; shortfall={report['shortfall']}", flush=True)
    return rows, report


require(SELECTED_BASELINE in {"B1", "B2", "B3"}, "SELECTED_BASELINE must be B1, B2, or B3.")
require(sum(bool(flag) for flag in (RUN_ALL_BASELINES, RUN_B1_REPEATS, RUN_B2_REPEATS)) <= 1,
        "Enable only one batch-run flag.")
if RUN_ALL_BASELINES:
    RUN_PLAN = [(setting, seed, condition) for setting in ("low_resource", "imbalanced")
                for seed in TRAINING_SEEDS for condition in ("B1", "B2", "B3")
                if not (setting == "low_resource" and condition == "B2")]
elif RUN_B2_REPEATS:
    RUN_PLAN = [("imbalanced", seed, "B2") for seed in TRAINING_SEEDS]
elif RUN_B1_REPEATS:
    RUN_PLAN = [(setting, seed, "B1") for setting in ("low_resource", "imbalanced")
                for seed in TRAINING_SEEDS]
else:
    RUN_PLAN = [(SELECTED_SETTING, SELECTED_SEED, SELECTED_BASELINE)]

completed_results = []
for setting, seed, condition in RUN_PLAN:
    if condition == "B2" and setting == "low_resource":
        require((RUN_ROOT / setting / "B1" / f"seed_{seed}" / "results.json").is_file(),
                "Balanced B2 is a no-op. Complete the corresponding B1 run first.")
        print(f"Balanced B2/seed_{seed} reuses B1; no training job is added.", flush=True)
        continue
    real_data, real_digest = load_real_subset(setting, seed)
    balancing_report = None
    if condition == "B3":
        train_rows, eda_report = prepare_eda(real_data, setting, seed, real_digest)
    elif condition == "B2":
        train_rows, balancing_report = build_random_oversampling(real_data, seed, NUM_LABELS)
        eda_report = None
    else:
        train_rows, eda_report = list(real_data), None
    signature = {
        "real_data_sha256": real_digest, "validation_sha256": VALIDATION_DIGEST,
        "actual_training_rows_sha256": rows_digest(train_rows),
        "eda_config": EDA_CONFIG if condition == "B3" else None,
    }
    if condition == "B2":
        signature["balancing"] = {
            key: balancing_report[key] for key in
            ("method", "implementation_version", "seed", "target_per_class", "real_unique_rows", "training_draws", "additional_draws")
        }
    completed_results.append(run_baseline(setting, seed, condition, real_data, train_rows, eda_report, signature,
                                          balancing_report=balancing_report))

balanced_b2_identities = record_balanced_b2_identities()


## 7. Review saved validation results

The main summary includes distinct completed training jobs, with mean and sample
standard deviation and the number of seeds available. Full baselines require 15
distinct fits: six B1, three imbalanced B2, and six B3. The three balanced B2
conditions reuse B1 and appear separately as identities, completing 18 logical
conditions without double-counting training jobs or compute.

`stage_log.csv` records EDA, training, validation, and checkpoint-validation time.
These are local elapsed observations. MPS hours are not billed T4 GPU hours; later
cost analysis must label the hardware and rate assumptions. `total_seconds` adds
original EDA generation time even when the augmentation is read from cache.

Compact B2 summaries and paired B1 comparisons are saved under `results/metrics/`
for version control. Checkpoints and detailed provenance remain in the local run folders.


In [ ]:
result_files = sorted(RUN_ROOT.glob("*/B*/seed_*/results.json"))
all_results = [json.loads(path.read_text()) for path in result_files]
require(all_results, "No baseline run has completed.")
summary_columns = ["setting", "condition", "seed", "macro_f1", "accuracy", "rare_macro_f1", "rare_macro_recall",
                   "best_epoch", "epochs_run", "real_rows", "train_rows", "real_unique_rows", "training_draws",
                   "balancing_method", "optimizer_steps", "augmentation_seconds", "training_seconds", "total_seconds", "device"]
summary_rows = [{**result, "real_unique_rows": result.get("real_unique_rows", result["real_rows"]),
                 "training_draws": result.get("training_draws", result["train_rows"]),
                 "balancing_method": result.get("balancing_method", "none")} for result in all_results]
summary = pd.DataFrame(summary_rows)[summary_columns].sort_values(["setting", "condition", "seed"])
summary.to_csv(RUN_ROOT / "validation_summary.csv", index=False)
aggregate = summary.groupby(["setting", "condition"]).agg(
    seeds_completed=("seed", "nunique"), macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
    accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    rare_f1_mean=("rare_macro_f1", "mean"), rare_f1_std=("rare_macro_f1", "std"),
    rare_recall_mean=("rare_macro_recall", "mean"), rare_recall_std=("rare_macro_recall", "std"),
    total_seconds_mean=("total_seconds", "mean"),
).reset_index()
aggregate.to_csv(RUN_ROOT / "validation_aggregate.csv", index=False)
display(summary)
display(aggregate)
print(f"Completed distinct baseline training jobs: {len(summary)}/15. Scores are validation results; official test data remains reserved.")
if balanced_b2_identities:
    display(pd.DataFrame(balanced_b2_identities))
    print(f"Balanced B2 identities: {len(balanced_b2_identities)}/3 reuse B1 and add no training jobs.")
b1_paths = [result["model_path"] for result in all_results if result["condition"] == "B1"]
print("B1 checkpoints available for F2:", *b1_paths, sep="\n")

metrics_dir = PROJECT_ROOT / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)
b2_results = summary[summary["condition"] == "B2"]
if not b2_results.empty:
    b2_results.to_csv(metrics_dir / "b2_validation_results.csv", index=False)
    aggregate[aggregate["condition"] == "B2"].to_csv(metrics_dir / "b2_validation_aggregate.csv", index=False)
    comparison_metrics = ["macro_f1", "accuracy", "rare_macro_f1", "rare_macro_recall", "total_seconds"]
    paired = summary[summary["condition"] == "B1"][["setting", "seed", *comparison_metrics]].merge(
        b2_results[["setting", "seed", *comparison_metrics]], on=["setting", "seed"],
        suffixes=("_B1", "_B2"), validate="one_to_one")
    for metric in comparison_metrics:
        paired[f"{metric}_B2_minus_B1"] = paired[f"{metric}_B2"] - paired[f"{metric}_B1"]
    paired.to_csv(metrics_dir / "b1_b2_validation_comparison.csv", index=False)
    display(paired)
    print("B2 summaries and paired B1 comparisons saved to:", metrics_dir)


References: [PyTorch MPS backend](https://docs.pytorch.org/docs/stable/notes/mps.html),
[DistilBERT](https://huggingface.co/docs/transformers/model_doc/distilbert),
[pinned pretrained model](https://huggingface.co/distilbert/distilbert-base-uncased/tree/12040accade4e8a0f71eabdb258fecc2e7e948be),
[macro-F1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html),
and [EDA paper](https://aclanthology.org/D19-1670/).